In [4]:
!pip install faiss-cpu sentence-transformers groq -q

import os
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer
from groq import Groq
from getpass import getpass

# --- NEW ADDITION: Mount Drive and set Working Directory ---
from google.colab import drive
drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/Hallucination_RAG_Project')
# -----------------------------------------------------------

# 1. Ensure API connection
if "GROQ_API_KEY" not in os.environ:
    print("Please enter your Groq API Key:")
    os.environ["GROQ_API_KEY"] = getpass()
client = Groq()

# 2. Load dataset and initialize retrieval engine
print("Initializing pipeline resources...")
df = pd.read_csv("./data/prototype_qa_data.csv")
# ... (rest of the code remains exactly the same)
knowledge_base = df['knowledge'].dropna().tolist()

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
document_embeddings = embedder.encode(knowledge_base, convert_to_numpy=True, normalize_embeddings=True)

dimension = document_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(document_embeddings)
print("Pipeline ready.\n")

# 3. System A: Baseline LLM
def run_system_a(query):
    try:
        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[
                {"role": "system", "content": "You are a helpful assistant. Answer directly and concisely."},
                {"role": "user", "content": query}
            ],
            temperature=0.0,
            max_tokens=150
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"Error: {e}"

# 4. System B: Standard Naive RAG
def run_system_b(query, k=2):
    query_vec = embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    _, indices = index.search(query_vec, k)

    retrieved_docs = [knowledge_base[i] for i in indices[0] if i != -1]
    context_str = "\n".join(retrieved_docs)

    prompt = f"Answer the question based strictly on the context.\n\nContext:\n{context_str}\n\nQuestion: {query}"
    try:
        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0,
            max_tokens=150
        )
        return response.choices[0].message.content.strip(), retrieved_docs
    except Exception as e:
        return f"Error: {e}", []

# 5. System C: Basic Filtered RAG
def run_system_c(query, k=3, threshold=0.45):
    query_vec = embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    scores, indices = index.search(query_vec, k)

    filtered_docs = []
    for score, idx in zip(scores[0], indices[0]):
        if idx != -1 and score >= threshold:
            filtered_docs.append((knowledge_base[idx], score))

    if not filtered_docs:
        context_str = "No sufficiently relevant context available."
    else:
        context_str = "\n".join([doc for doc, _ in filtered_docs])

    prompt = f"Answer the question based strictly on the context.\n\nContext:\n{context_str}\n\nQuestion: {query}"
    try:
        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0,
            max_tokens=150
        )
        return response.choices[0].message.content.strip(), filtered_docs
    except Exception as e:
        return f"Error: {e}", []

# 6. Unified Comparison Function
def compare_all_systems(query):
    print("=" * 70)
    print(f"QUERY: {query}")
    print("=" * 70)

    # Run System A
    ans_a = run_system_a(query)
    print("\n[SYSTEM A — BASELINE LLM]")
    print(f"Answer: {ans_a}")

    # Run System B
    ans_b, docs_b = run_system_b(query, k=2)
    print("\n[SYSTEM B — NAIVE RAG]")
    print(f"Retrieved Chunks: {len(docs_b)}")
    print(f"Answer: {ans_b}")

    # Run System C
    ans_c, docs_c = run_system_c(query, k=3, threshold=0.45)
    print("\n[SYSTEM C — PROPOSED BASIC RAG]")
    print(f"Filtered Chunks Passing Threshold: {len(docs_c)}")
    for i, (text, score) in enumerate(docs_c):
        print(f" - Chunk {i+1} (Score: {score:.3f}): {text[:90]}...")
    print(f"Answer: {ans_c}")
    print("=" * 70)

# Run test on sample question
sample_q = df.iloc[0]['question']
compare_all_systems(sample_q)

Mounted at /content/drive
Initializing pipeline resources...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Pipeline ready.

QUERY: Which magazine was started first Arthur's Magazine or First for Women?

[SYSTEM A — BASELINE LLM]
Answer: First for Women was started in 1995. I couldn't find information on Arthur's Magazine being a well-known publication.

[SYSTEM B — NAIVE RAG]
Retrieved Chunks: 2
Answer: Based on the context, Arthur's Magazine (1844–1846) was published first, as it was published from 1844 to 1846, while the information about First for Women does not provide a specific start date.

[SYSTEM C — PROPOSED BASIC RAG]
Filtered Chunks Passing Threshold: 1
 - Chunk 1 (Score: 0.875): Arthur's Magazine (1844–1846) was an American literary periodical published in Philadelphi...
Answer: Based on the context, Arthur's Magazine was started first, in 1844, while First for Women was not mentioned with a specific start date in the given context.
